pydantic base model ()

class Prompter:

    def __init__(self, db_path: str, config:Dict [str]): (dbpath
        keep dbpath if end with .tmp else make dbpath with .tmp ending
        config = config
        create duckdb con
        make sure columns have right columns like status etc

    __exit__


    setup db
        check status column

    async send single prompt
        make sure status is not complete

    resume/start loop
        start at row with no status and gather them into transactions to do
        -- use ACID for sql transactions 
            Atomicity: A transaction is either fully completed or fully rolled back. If any part fails, the entire transaction fails.
            Consistency: A transaction moves the database from one valid state to another while following all rules and constraint.
            Isolation: Transactions run independently of each other, even when executed at the same time.
            Durability: Once a transaction is committed, its changes remain permanent, even after a system failure

    finalize
        change db file name from .duckdb.tmp -> .duckdb once all status's are complete
        


    
    

In [69]:
import asyncio
from threading import Thread, current_thread
import duckdb
import requests
from pydantic import BaseModel, ValidationError, BeforeValidator, field_validator
from typing import Optional, List, Dict, Any
from pathlib import Path
from enum import StrEnum
import re

# https://duckdb.org/docs/stable/guides/python/multiple_threads
# https://duckdb.org/docs/stable/sql/statements/transactions
# https://en.wikipedia.org/wiki/Database_transaction

# https://pmc.ncbi.nlm.nih.gov/articles/PMC9081216/

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "tinyllama"
DATA_PATH = "../data/raw/example_variant_data.csv"

prompt = """You are an expert system.
    Analyze the following somatic variant data and predict its pathogenicity
    (do not rely on any previous examples or context):
    
    ### INPUT DATA
    Chromosome: 3
    ChromosomePosition: 178937422
    Ref: T
    RefAA: Cys
    Alt: C
    AltAA: Arg
    Type: missense
    Coverage: 806
    Gene: PIK3CA
    GeneStrand: +
    ExonNumber: 12
    Transcript: NM_006218.2
    Protein: NP_006209.2
    CodingBase: 1810
    CodonPosition: 1
    AAPosition: 604
    HGVSGenomic: g.178937422T>C
    HGVSCodingTranscript: NM_006218.2
    HGVSCoding: c.1810T>C
    HGVSTranslationProtein: NP_006209.2
    HGVSProtein: p.Cys604Arg
    ### TASK
    Based on this information, please classify the variant as 'Benign', 'Likely Benign', 'Likely Pathogenic',
    'Pathogenic', or 'Unknown Significance'. Answer with only one type."""

prompt_test = 'Hello!'


class Classification(StrEnum):
    BENIGN = 'Benign'
    LIKELY_BENIGN = 'Likely Benign'
    PATHOGENIC = 'Pathogenic'
    LIKELY_PATHOGENIC = 'Likely Pathogenic'
    VUS = 'VUS'

class Pathogenicity(BaseModel):
    classification: Classification

    @field_validator('classification', mode='before')
    @classmethod
    def parse(cls, value: str | Classification) -> Classification:
        if isinstance(value, Classification):
            return value

        text = re.sub(r'[^a-z0-9]', ' ', str(value).lower()).strip()

        mapping = {
            # '[ \-_]*' means zero or more spaces, dashes, or underscores
            r'variant[ \-_]*of[ \-_]*uncertain[ \-_]*significance': Classification.VUS,
            r'variant[ \-_]*of[ \-_]*unknown[ \-_]*significance': Classification.VUS,
            r'uncertain[ \-_]*significance': Classification.VUS,
            r'unknown[ \-_]*significance': Classification.VUS,
            
            r'likely[ \-_]*pathogenic': Classification.LIKELY_PATHOGENIC,
            r'likely[ \-_]*oncogenic': Classification.LIKELY_PATHOGENIC,
            
            r'likely[ \-_]*benign': Classification.LIKELY_BENIGN,
            
            # No regex needed for single words
            r'pathogenic': Classification.PATHOGENIC,
            r'oncogenic': Classification.PATHOGENIC,
            r'benign': Classification.BENIGN,
            r'vus': Classification.VUS,
        }

        found = set()

        sorted_patterns = sorted(mapping.keys(), key=len, reverse=True)
        
        for pattern in  sorted_patterns:
            regex = re.compile(rf'\b{pattern}\b')
            if regex.search(text):
                found.add(mapping[pattern])
                text = regex.sub('', text)

        print(found)
        
        if len(found) == 1:
            return list(found)[0]
        elif len(found) > 1:
            raise ValueError(f"Multiple classifications detected: {found} '{value}'")
        else:    
            raise ValueError(f"Could not map '{value}' to a valid Classification")

class Prompter:
    """
    This is a reST style.
    
    :param param1: this is a first param
    :param param2: this is a second param
    :returns: this is a description of what is returned
    :raises keyError: raises an exception
    """


    def __init__(self, db_path: str='test', config: Path = 'test'):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        self.db_path = db_path if db_path.endswith(".tmp") else f"{db_path}.tmp"
        self.config = config
        self.conn = duckdb.connect(self.db_path)
        self.queue = asyncio.Queue()
        self._setup_db()

    def _setup_db(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        # SQL logic to ensure 'status' (pending/completed/failed) columns exist
        pass

    def send_single_prompt(self, row_id: str, prompt: str) -> Optional[LLMResponse]:
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception

        make new duckdb.con for each prompt
        """
        try:
            # Async HTTP request logic here
            # response = await client.post(...)
            
            # Runtime Validation
            # return LLMResponse.model_validate(response.json())
            response = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=60
            )

            result = response.json()["response"] # extract LLM response from JSON object
            print(result)
            pass
        except Exception as e:
            # Log error and return None so status can be marked 'failed'
            print(e)
            return None

    async def resume_loop(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        # 1. Query: SELECT * FROM data WHERE status != 'completed'
        # 2. Gather tasks for send_single_prompt
        # 3. Use Transactional Commits (BEGIN/COMMIT) for batch writes
        pass

    def finalize(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        pass

In [44]:
# --- Test Cases ---

# Case 1: The "No Spaces" / CamelCase LLM Hallucination
print(Pathogenicity(classification="LikelyPathogenic"))
# Output: classification=<Classification.LIKELY_PATHOGENIC: 'likelypathogenic'>

# Case 2: The "Forgot Spaces" inside a Paragraph
print(Pathogenicity(classification="Review suggests this is LikelyBenign based on frequency."))
# Output: classification=<Classification.LIKELY_BENIGN: 'likelybenign'>

# Case 3: Standard Spacing
print(Pathogenicity(classification="Likely Pathogenic"))
# Output: classification=<Classification.LIKELY_PATHOGENIC: 'likelypathogenic'>

# Case 4: Long Squashed Phrase
print(Pathogenicity(classification="variantofuncertainsignificance"))
# Output: classification=<Classification.VUS: 'vus'>

print(Pathogenicity(classification="Variant of uncertain significance"))

print(Pathogenicity(classification="Classification: 'Pathogenic'"))

{<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>}
classification=<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>
{<Classification.LIKELY_BENIGN: 'Likely Benign'>}
classification=<Classification.LIKELY_BENIGN: 'Likely Benign'>
{<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>}
classification=<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>
{<Classification.VUS: 'VUS'>}
classification=<Classification.VUS: 'VUS'>
{<Classification.VUS: 'VUS'>}
classification=<Classification.VUS: 'VUS'>
{<Classification.PATHOGENIC: 'Pathogenic'>}
classification=<Classification.PATHOGENIC: 'Pathogenic'>


In [42]:
print(Pathogenicity(classification="Pathogenic Pathogenic"))
print(Pathogenicity(classification="PathogenicPathogenic"))
print(Pathogenicity(classification="Im not sure but its probably not benign but pathogenic benign'"))

{<Classification.PATHOGENIC: 'Pathogenic'>}
classification=<Classification.PATHOGENIC: 'Pathogenic'>
set()


ValidationError: 1 validation error for Pathogenicity
classification
  Value error, Could not map 'PathogenicPathogenic' to a valid Classification [type=value_error, input_value='PathogenicPathogenic', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [70]:
p = Prompter()
print(p.send_single_prompt(1, prompt_test))

Yes, I am a helpful AI assistant. I'm always here to assist you with your queries and provide you with the best possible solutions. Whether it's about your daily routine or something more complex, I'm here to help you in any way I can. So don't hesitate to reach out to me whenever you need my assistance. I'm always ready to listen and offer you the best possible solution. Thank you for choosing me as your AI assistant!
None
